# Attention Mechanisms - Examples

Comprehensive examples of attention mechanisms for deep learning.

In [ ]:
import numpy as np
from typing import Tuple, Optional, List, Dict, Callable
import warnings
warnings.filterwarnings('ignore')

## 1. Basic Attention Mechanism

Implement basic attention with different scoring functions.

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

# Dot-product attention score
def dot_product_score(query: np.ndarray, keys: np.ndarray) -> np.ndarray:
    """query: (d_k,), keys: (n_k, d_k) -> (n_k,)"""
    return query @ keys.T

# Scaled dot-product score
def scaled_dot_product_score(query: np.ndarray, keys: np.ndarray) -> np.ndarray:
    d_k = keys.shape[-1]
    return (query @ keys.T) / np.sqrt(d_k)

# Additive (Bahdanau) attention score
def additive_score(query: np.ndarray, keys: np.ndarray,
                   W_q: np.ndarray, W_k: np.ndarray, v: np.ndarray) -> np.ndarray:
    q_transformed = W_q @ query
    k_transformed = keys @ W_k.T
    combined = np.tanh(q_transformed + k_transformed)
    return combined @ v

# Basic attention computation
def attention(query: np.ndarray, keys: np.ndarray, values: np.ndarray,
              score_fn: Callable) -> Tuple[np.ndarray, np.ndarray]:
    """Compute attention output and weights."""
    scores = score_fn(query, keys)
    weights = softmax(scores)
    output = weights @ values
    return output, weights

In [ ]:
# Example: simple sentence attention
np.random.seed(42)
n_tokens = 5
d_k = 4
d_v = 3

keys = np.random.randn(n_tokens, d_k)
values = np.random.randn(n_tokens, d_v)
query = np.random.randn(d_k)

print("Input Shapes:")
print(f"  Query: {query.shape}")
print(f"  Keys: {keys.shape}")
print(f"  Values: {values.shape}")

print("\nAttention with Different Scoring Functions:")

output_dot, weights_dot = attention(query, keys, values, dot_product_score)
print(f"\n  Dot Product:")
print(f"    Weights: {np.round(weights_dot, 4)}")
print(f"    Output: {np.round(output_dot, 4)}")

output_scaled, weights_scaled = attention(query, keys, values, scaled_dot_product_score)
print(f"\n  Scaled Dot Product:")
print(f"    Weights: {np.round(weights_scaled, 4)}")
print(f"    Output: {np.round(output_scaled, 4)}")

# Additive attention
d_hidden = 8
W_q = np.random.randn(d_hidden, d_k) * 0.1
W_k = np.random.randn(d_hidden, d_k) * 0.1
v = np.random.randn(d_hidden) * 0.1

additive_fn = lambda q, k: additive_score(q, k, W_q, W_k, v)
output_add, weights_add = attention(query, keys, values, additive_fn)
print(f"\n  Additive (Bahdanau):")
print(f"    Weights: {np.round(weights_add, 4)}")
print(f"    Output: {np.round(output_add, 4)}")

## 2. Scaled Dot-Product Attention

Implement scaled dot-product attention as used in Transformers.

In [ ]:
def scaled_dot_product_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    mask: Optional[np.ndarray] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Scaled Dot-Product Attention.
    
    Q: Queries (..., n_q, d_k)
    K: Keys (..., n_k, d_k)
    V: Values (..., n_k, d_v)
    mask: Optional mask (..., n_q, n_k)
    
    Returns: (output, attention_weights)
    """
    d_k = K.shape[-1]
    
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)
    
    if mask is not None:
        scores = scores + mask
    
    attention_weights = softmax(scores, axis=-1)
    output = attention_weights @ V
    
    return output, attention_weights

# Example usage
np.random.seed(42)
batch_size = 2
n_q = 4
n_k = 6
d_k = 8
d_v = 8

Q = np.random.randn(batch_size, n_q, d_k)
K = np.random.randn(batch_size, n_k, d_k)
V = np.random.randn(batch_size, n_k, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)

print("Scaled Dot-Product Attention:")
print(f"  Q shape: {Q.shape}")
print(f"  K shape: {K.shape}")
print(f"  V shape: {V.shape}")
print(f"  Output shape: {output.shape}")
print(f"  Weights shape: {weights.shape}")
print(f"  Weights sum (should be 1): {np.round(weights[0, 0].sum(), 4)}")

In [ ]:
# Why scaling matters
print("Why Scaling Matters (d_k effect):")

for d in [8, 64, 512]:
    q = np.random.randn(d)
    k = np.random.randn(d)
    
    dot = np.dot(q, k)
    scaled = dot / np.sqrt(d)
    
    print(f"  d_k={d:3d}: dot={dot:8.2f}, scaled={scaled:8.2f}")

print("\n  Large d_k → large dot products → saturated softmax → tiny gradients")
print("  Scaling keeps variance ~1, preventing saturation")

## 3. Multi-Head Attention

Implement multi-head attention from Transformer architecture.

In [ ]:
class MultiHeadAttention:
    """
    Multi-Head Attention module.
    
    MultiHead(Q, K, V) = Concat(head_1, ..., head_h)W^O
    where head_i = Attention(QW^Q_i, KW^K_i, VW^V_i)
    """
    
    def __init__(self, d_model: int, n_heads: int, seed: int = 42):
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        np.random.seed(seed)
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_v = d_model // n_heads
        
        scale = np.sqrt(2.0 / d_model)
        self.W_q = np.random.randn(d_model, d_model) * scale
        self.W_k = np.random.randn(d_model, d_model) * scale
        self.W_v = np.random.randn(d_model, d_model) * scale
        self.W_o = np.random.randn(d_model, d_model) * scale
    
    def split_heads(self, x: np.ndarray) -> np.ndarray:
        """(batch, seq_len, d_model) -> (batch, n_heads, seq_len, d_k)"""
        batch_size, seq_len, _ = x.shape
        x = x.reshape(batch_size, seq_len, self.n_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)
    
    def combine_heads(self, x: np.ndarray) -> np.ndarray:
        """(batch, n_heads, seq_len, d_k) -> (batch, seq_len, d_model)"""
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(0, 2, 1, 3)
        return x.reshape(batch_size, seq_len, self.d_model)
    
    def forward(self, Q: np.ndarray, K: np.ndarray, V: np.ndarray,
                mask: Optional[np.ndarray] = None) -> Tuple[np.ndarray, np.ndarray]:
        batch_size = Q.shape[0]
        
        Q_proj = Q @ self.W_q
        K_proj = K @ self.W_k
        V_proj = V @ self.W_v
        
        Q_heads = self.split_heads(Q_proj)
        K_heads = self.split_heads(K_proj)
        V_heads = self.split_heads(V_proj)
        
        scores = Q_heads @ K_heads.swapaxes(-2, -1) / np.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores + mask
        
        attention_weights = softmax(scores, axis=-1)
        head_outputs = attention_weights @ V_heads
        
        concat = self.combine_heads(head_outputs)
        output = concat @ self.W_o
        
        return output, attention_weights

In [ ]:
# Example usage
np.random.seed(42)
batch_size = 2
seq_len = 6
d_model = 64
n_heads = 8

mha = MultiHeadAttention(d_model, n_heads)
X = np.random.randn(batch_size, seq_len, d_model)

output, weights = mha.forward(X, X, X)  # Self-attention

print("Multi-Head Attention:")
print(f"  d_model: {d_model}")
print(f"  n_heads: {n_heads}")
print(f"  d_k (per head): {mha.d_k}")
print(f"\n  Input shape: {X.shape}")
print(f"  Output shape: {output.shape}")
print(f"  Attention weights shape: {weights.shape}")

# Analyze attention patterns
print("\nAttention Pattern Analysis:")
print(f"  First head, first batch:")
print(f"  {np.round(weights[0, 0], 3)}")

print("\n  Entropy per head (first batch, first query):")
for h in range(n_heads):
    w = weights[0, h, 0]
    entropy = -np.sum(w * np.log(w + 1e-10))
    print(f"    Head {h}: entropy = {entropy:.4f}")

## 4. Self-Attention and Cross-Attention

Demonstrate self-attention and cross-attention patterns.

In [ ]:
def attention_simple(Q, K, V, mask=None):
    d_k = K.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

np.random.seed(42)
d_model = 16

# Self-attention example
print("1. Self-Attention:")
print("   Q, K, V all come from the same sequence")

seq_len = 5
X = np.random.randn(seq_len, d_model)

W_q = np.random.randn(d_model, d_model) * 0.1
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1

Q = X @ W_q
K = X @ W_k
V = X @ W_v

output, weights = attention_simple(Q, K, V)

print(f"\n   Input sequence: {X.shape}")
print(f"   Self-attention weights:")
print(f"   {np.round(weights, 3)}")

In [ ]:
# Cross-attention example
print("2. Cross-Attention:")
print("   Q from one sequence, K/V from another (encoder-decoder)")

encoder_len = 4
decoder_len = 3

encoder_output = np.random.randn(encoder_len, d_model)
decoder_state = np.random.randn(decoder_len, d_model)

Q = decoder_state @ W_q  # Queries from decoder
K = encoder_output @ W_k  # Keys from encoder
V = encoder_output @ W_v  # Values from encoder

output, weights = attention_simple(Q, K, V)

print(f"\n   Encoder (source): {encoder_output.shape}")
print(f"   Decoder (target): {decoder_state.shape}")
print(f"   Cross-attention weights (decoder → encoder):")
print(f"   {np.round(weights, 3)}")
print(f"   Shape: {weights.shape} (decoder_len × encoder_len)")

print("\n   Interpretation:")
print("   - Row i: which encoder positions decoder position i attends to")
print("   - Useful for alignment in translation")

## 5. Attention Masking

Implement padding and causal masking for attention.

In [ ]:
def attention_with_mask(Q, K, V, mask=None):
    d_k = K.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

def create_causal_mask(seq_len: int) -> np.ndarray:
    """Create lower-triangular mask for causal attention."""
    mask = np.triu(np.ones((seq_len, seq_len)), k=1)
    mask = np.where(mask == 1, -np.inf, 0)
    return mask

def create_padding_mask(lengths: np.ndarray, max_len: int) -> np.ndarray:
    """Create mask for padded sequences."""
    batch_size = len(lengths)
    mask = np.zeros((batch_size, max_len))
    for i, length in enumerate(lengths):
        mask[i, length:] = 1
    mask = np.where(mask == 1, -np.inf, 0)
    return mask[:, np.newaxis, np.newaxis, :]

np.random.seed(42)
seq_len = 5
d_k = 8

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

# No mask (bidirectional)
print("1. No Mask (Bidirectional Attention):")
_, weights_bi = attention_with_mask(Q, K, V)
print(f"   Attention weights:")
for row in np.round(weights_bi, 3):
    print(f"   {row}")

In [ ]:
# Causal mask
print("2. Causal Mask (Autoregressive):")
causal_mask = create_causal_mask(seq_len)
print(f"   Causal mask (0=allow, -inf=block):")
mask_visual = np.where(causal_mask == -np.inf, 'X', '0')
for row in mask_visual:
    print(f"   {''.join(row)}")

_, weights_causal = attention_with_mask(Q, K, V, causal_mask)
print(f"\n   Attention weights (causal):")
for row in np.round(weights_causal, 3):
    print(f"   {row}")
print("   Note: upper triangle is zero (can't attend to future)")

In [ ]:
# Padding mask
print("3. Padding Mask:")
batch_size = 2
max_len = 6
lengths = np.array([4, 3])

Q_batch = np.random.randn(batch_size, max_len, d_k)
K_batch = np.random.randn(batch_size, max_len, d_k)
V_batch = np.random.randn(batch_size, max_len, d_k)

padding_mask = create_padding_mask(lengths, max_len)
print(f"   Lengths: {lengths}")
print(f"   Padding mask shape: {padding_mask.shape}")

_, weights_padded = attention_with_mask(Q_batch, K_batch, V_batch, padding_mask)
print(f"\n   Batch 0, first query (attends to pos 0-3):")
print(f"   {np.round(weights_padded[0, 0], 4)}")
print(f"   Batch 1, first query (attends to pos 0-2):")
print(f"   {np.round(weights_padded[1, 0], 4)}")

## 6. Positional Encoding

Implement sinusoidal and learned positional encodings.

In [ ]:
def sinusoidal_positional_encoding(max_len: int, d_model: int) -> np.ndarray:
    """
    Sinusoidal positional encoding from "Attention is All You Need".
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    PE = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    
    PE[:, 0::2] = np.sin(position * div_term)
    PE[:, 1::2] = np.cos(position * div_term)
    
    return PE

class LearnedPositionalEncoding:
    """Learned positional embeddings."""
    
    def __init__(self, max_len: int, d_model: int, seed: int = 42):
        np.random.seed(seed)
        self.embeddings = np.random.randn(max_len, d_model) * 0.02
    
    def forward(self, seq_len: int) -> np.ndarray:
        return self.embeddings[:seq_len]

# Demonstrate sinusoidal encoding
max_len = 100
d_model = 64

PE = sinusoidal_positional_encoding(max_len, d_model)

print("Sinusoidal Positional Encoding:")
print(f"  Shape: {PE.shape}")
print(f"  PE[0, :8]: {np.round(PE[0, :8], 4)}")
print(f"  PE[1, :8]: {np.round(PE[1, :8], 4)}")

print("\n  Periodicity:")
print(f"    Dim 0: PE[0,0]={PE[0,0]:.4f}, PE[3,0]={PE[3,0]:.4f}, PE[6,0]={PE[6,0]:.4f}")
print(f"    Dim 6: PE[0,6]={PE[0,6]:.4f}, PE[3,6]={PE[3,6]:.4f}, PE[6,6]={PE[6,6]:.4f}")

In [ ]:
# Rotary Position Embedding (RoPE)
def rope_encoding(x: np.ndarray, positions: np.ndarray) -> np.ndarray:
    """Apply Rotary Position Embedding."""
    seq_len, d = x.shape
    theta = 10000 ** (-2 * np.arange(d // 2) / d)
    
    angles = positions[:, np.newaxis] * theta[np.newaxis, :]
    x_pairs = x.reshape(seq_len, d // 2, 2)
    
    cos_angles = np.cos(angles)[:, :, np.newaxis]
    sin_angles = np.sin(angles)[:, :, np.newaxis]
    
    x_rotated = np.zeros_like(x_pairs)
    x_rotated[:, :, 0] = x_pairs[:, :, 0] * cos_angles[:, :, 0] - x_pairs[:, :, 1] * sin_angles[:, :, 0]
    x_rotated[:, :, 1] = x_pairs[:, :, 0] * sin_angles[:, :, 0] + x_pairs[:, :, 1] * cos_angles[:, :, 0]
    
    return x_rotated.reshape(seq_len, d)

# RoPE demonstration
print("Rotary Position Embedding (RoPE):")
seq_len = 5
d = 8
x = np.random.randn(seq_len, d)
positions = np.arange(seq_len)

x_rope = rope_encoding(x, positions)
print(f"  Input shape: {x.shape}")
print(f"  After RoPE: {x_rope.shape}")
print("  Note: Encodes relative position in dot products")

# Learned encoding
print("\nLearned Positional Encoding:")
learned_pe = LearnedPositionalEncoding(max_len, d_model)
print(f"  Shape: {learned_pe.embeddings.shape}")
print(f"  Note: Can't extrapolate beyond max_len")

## 7. Attention Analysis

Analyze attention patterns and their properties.

In [ ]:
def attention_entropy(weights: np.ndarray) -> np.ndarray:
    """Low entropy = focused, High entropy = diffuse."""
    return -np.sum(weights * np.log(weights + 1e-10), axis=-1)

def attention_effective_context(weights: np.ndarray, threshold: float = 0.01) -> np.ndarray:
    """Count positions with attention weight above threshold."""
    return np.sum(weights > threshold, axis=-1)

def attention_mean_distance(weights: np.ndarray) -> np.ndarray:
    """Compute mean attended distance from each position."""
    seq_len = weights.shape[-1]
    positions = np.arange(seq_len)
    
    mean_dist = np.zeros(weights.shape[:-1])
    for i in range(seq_len):
        distances = np.abs(positions - i)
        mean_dist[..., i] = np.sum(weights[..., i, :] * distances, axis=-1)
    
    return mean_dist

# Create attention patterns
np.random.seed(42)
seq_len = 10
d_k = 16

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)

scores = Q @ K.T / np.sqrt(d_k)
weights_uniform = softmax(scores)
weights_focused = softmax(scores * 2)
weights_hard = softmax(scores * 10)

print("Attention Pattern Analysis:")

for name, weights in [
    ("Normal", weights_uniform),
    ("Focused (×2)", weights_focused),
    ("Hard (×10)", weights_hard)
]:
    entropy = attention_entropy(weights)
    eff_context = attention_effective_context(weights)
    mean_dist = attention_mean_distance(weights)
    
    print(f"\n  {name}:")
    print(f"    Mean entropy: {np.mean(entropy):.4f}")
    print(f"    Mean effective context: {np.mean(eff_context):.2f} positions")
    print(f"    Mean attention distance: {np.mean(mean_dist):.2f}")

## 8. Linear Attention

Implement linear attention for efficient long sequences.

In [ ]:
def standard_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray) -> np.ndarray:
    """Standard scaled dot-product. Complexity: O(n² d)"""
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    return weights @ V

def elu_feature(x: np.ndarray) -> np.ndarray:
    """ELU feature map for linear attention."""
    return np.where(x > 0, x + 1, np.exp(x))

def linear_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray,
                     feature_fn: callable = None) -> np.ndarray:
    """
    Linear attention using feature maps.
    Complexity: O(n d²) instead of O(n² d)
    """
    if feature_fn is None:
        feature_fn = elu_feature
    
    Q_prime = feature_fn(Q)
    K_prime = feature_fn(K)
    
    KTV = K_prime.T @ V  # O(n_k d d_v)
    numerator = Q_prime @ KTV  # O(n_q d d_v)
    
    K_sum = K_prime.sum(axis=0)
    denominator = Q_prime @ K_sum
    
    return numerator / (denominator[:, np.newaxis] + 1e-10)

# Compare complexity
print("Complexity Comparison:")
for seq_len in [64, 256, 1024]:
    d = 64
    standard_ops = seq_len * seq_len * d + seq_len * seq_len * d
    linear_ops = seq_len * d * d + seq_len * d * d
    
    print(f"  n={seq_len:4d}, d={d}: Standard={standard_ops/1e6:.1f}M, Linear={linear_ops/1e6:.1f}M, Ratio={standard_ops/linear_ops:.1f}x")

In [ ]:
# Accuracy comparison
seq_len = 32
d = 16
np.random.seed(42)
Q = np.random.randn(seq_len, d) * 0.5
K = np.random.randn(seq_len, d) * 0.5
V = np.random.randn(seq_len, d)

output_standard = standard_attention(Q, K, V)
output_linear = linear_attention(Q, K, V)

print(f"Accuracy Comparison (seq_len={seq_len}, d={d}):")
print(f"  Standard output mean: {np.mean(output_standard):.4f}")
print(f"  Linear output mean: {np.mean(output_linear):.4f}")
print(f"  MSE: {np.mean((output_standard - output_linear)**2):.6f}")

## 9. Sparse Attention Patterns

Implement various sparse attention patterns.

In [ ]:
def create_local_mask(seq_len: int, window: int) -> np.ndarray:
    """Local (sliding window) attention mask."""
    mask = np.full((seq_len, seq_len), -np.inf)
    for i in range(seq_len):
        start = max(0, i - window)
        end = min(seq_len, i + window + 1)
        mask[i, start:end] = 0
    return mask

def create_strided_mask(seq_len: int, stride: int) -> np.ndarray:
    """Strided attention: attend to every stride-th position."""
    mask = np.full((seq_len, seq_len), -np.inf)
    for i in range(seq_len):
        mask[i, ::stride] = 0
        mask[i, i] = 0
    return mask

def create_block_sparse_mask(seq_len: int, block_size: int) -> np.ndarray:
    """Block-sparse attention: attend within blocks."""
    mask = np.full((seq_len, seq_len), -np.inf)
    n_blocks = seq_len // block_size
    
    for b in range(n_blocks):
        start = b * block_size
        end = start + block_size
        mask[start:end, start:end] = 0
    
    remainder_start = n_blocks * block_size
    if remainder_start < seq_len:
        mask[remainder_start:, remainder_start:] = 0
    
    return mask

def create_longformer_mask(seq_len: int, window: int, 
                           global_indices: List[int]) -> np.ndarray:
    """Longformer-style: local + global attention."""
    mask = create_local_mask(seq_len, window)
    
    for idx in global_indices:
        mask[idx, :] = 0
        mask[:, idx] = 0
    
    return mask

seq_len = 16
print("Sparse Attention Patterns:")

patterns = [
    ("Local (w=2)", create_local_mask(seq_len, window=2)),
    ("Strided (s=4)", create_strided_mask(seq_len, stride=4)),
    ("Block (b=4)", create_block_sparse_mask(seq_len, block_size=4)),
    ("Longformer (w=2, g=[0,8])", create_longformer_mask(seq_len, window=2, global_indices=[0, 8])),
]

for name, mask in patterns:
    density = np.mean(mask > -np.inf)
    print(f"\n  {name}:")
    print(f"    Density: {density:.2%}")
    
    pattern_str = np.where(mask > -np.inf, '.', 'x')
    for i, row in enumerate(pattern_str[:6]):
        print(f"    {''.join(row[:8])}...")

## 10. Complete Transformer Block

Implement a complete transformer encoder block.

In [ ]:
def gelu(x: np.ndarray) -> np.ndarray:
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

class LayerNorm:
    def __init__(self, d_model: int, eps: float = 1e-6):
        self.eps = eps
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        mean = np.mean(x, axis=-1, keepdims=True)
        std = np.std(x, axis=-1, keepdims=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

class FeedForward:
    def __init__(self, d_model: int, d_ff: int, seed: int = 42):
        np.random.seed(seed)
        scale = np.sqrt(2.0 / d_model)
        self.W1 = np.random.randn(d_model, d_ff) * scale
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * scale
        self.b2 = np.zeros(d_model)
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        hidden = gelu(x @ self.W1 + self.b1)
        return hidden @ self.W2 + self.b2

class TransformerEncoderBlock:
    def __init__(self, d_model: int, n_heads: int, d_ff: int, seed: int = 42):
        self.attention = MultiHeadAttention(d_model, n_heads, seed)
        self.norm1 = LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, seed + 1)
        self.norm2 = LayerNorm(d_model)
    
    def forward(self, x: np.ndarray, mask: np.ndarray = None) -> np.ndarray:
        attn_output, _ = self.attention.forward(x, x, x, mask)
        x = self.norm1.forward(x + attn_output)
        ffn_output = self.ffn.forward(x)
        x = self.norm2.forward(x + ffn_output)
        return x

In [ ]:
class TransformerEncoder:
    def __init__(self, n_layers: int, d_model: int, n_heads: int, 
                 d_ff: int, max_len: int, seed: int = 42):
        self.layers = [
            TransformerEncoderBlock(d_model, n_heads, d_ff, seed=seed + i)
            for i in range(n_layers)
        ]
        self.pos_encoding = sinusoidal_positional_encoding(max_len, d_model)
    
    def forward(self, x: np.ndarray, mask: np.ndarray = None) -> np.ndarray:
        seq_len = x.shape[1]
        x = x + self.pos_encoding[:seq_len]
        for layer in self.layers:
            x = layer.forward(x, mask)
        return x

# Create and test transformer
np.random.seed(42)

n_layers = 4
d_model = 128
n_heads = 8
d_ff = 512
max_len = 256

encoder = TransformerEncoder(n_layers, d_model, n_heads, d_ff, max_len)

batch_size = 2
seq_len = 32
x = np.random.randn(batch_size, seq_len, d_model)

output = encoder.forward(x)

print("Transformer Encoder Configuration:")
print(f"  Layers: {n_layers}")
print(f"  d_model: {d_model}")
print(f"  n_heads: {n_heads}")
print(f"  d_ff: {d_ff}")

print(f"\nInput/Output:")
print(f"  Input shape: {x.shape}")
print(f"  Output shape: {output.shape}")

# Parameter count
params_attention = 4 * d_model * d_model
params_ffn = d_model * d_ff + d_ff + d_ff * d_model + d_model
params_norm = 2 * d_model * 2
params_per_layer = params_attention + params_ffn + params_norm
total_params = n_layers * params_per_layer

print(f"\nParameter Count:")
print(f"  Per layer: {params_per_layer:,}")
print(f"  Total: {total_params:,}")

print("\nAll attention mechanism examples completed!")